In [0]:
%sql
CREATE OR REPLACE TABLE taxi_labs.new_data.new_bronze_taxi TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported') AS
SELECT VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance,
       RatecodeID, store_and_fwd_flag, PULocationID, DOLocationID, payment_type, fare_amount,
       extra, mta_tax, tip_amount, tolls_amount, improvement_surcharge, total_amount,
       congestion_surcharge, Airport_fee, cbd_congestion_fee
FROM taxi_labs.pipelines.taxi_data;

In [0]:
%sql
create or REPLACE TABLE taxi_labs.new_data.taxi_data_clean (
    CONSTRAINT valid_fare EXPECT (fare_amount IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_distance EXPECT (trip_distance IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_times EXPECT (tpep_pickup_datetime >= tpep_dropoff_datetime) ON VIOLATION DROP ROW, 
    CONSTRAINT valid_zone EXPECT (PUlocationID BETWEEN 0 AND 265) ON VIOLATION DROP ROW,
    CONSTRAINT zone_exists EXPECT (PUlocationID IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT has_fare EXPECT (fare_amount IS NOT NULL) ON VIOLATION DROP ROW) as select b.*, c.zone as pickup_zone 
    from (select distinct * from taxi_labs.pipelines.bronze_taxi) b
    left join taxi_labs.pipelines.taxizones_silver c on b.PUlocationID = c.LocationID 

In [0]:
%sql 
CREATE OR REPLACE TABLE taxi_labs.new_data.taxi_data_quarantined as 
select *, concat_ws(',',
    CASE WHEN NOT COALESCE(fare_amount >= 0, FALSE) THEN 'valid_fare' END,
    CASE WHEN NOT COALESCE(trip_distance BETWEEN 0 AND 500, FALSE) THEN 'valid_distance' END,
    CASE WHEN NOT COALESCE(tpep_pickup_datetime < tpep_dropoff_datetime, FALSE) THEN 'valid_times' END
) AS failed_rules
from (select distinct * from taxi_labs.pipelines.bronze_taxi)
WHERE NOT (coalesce(fare_amount >= 0, false)
       AND coalesce(trip_distance BETWEEN 0 AND 500, false)
       AND coalesce(tpep_dropoff_datetime >= tpep_pickup_datetime, false)); 



In [0]:
CREATE OR REPLACE TABLE taxi_labs.new_data.silver_view AS SELECT ROW_NUMBER() OVER (ORDER BY tpep_pickup_datetime) as trip_id, VendorID, tpep_pickup_datetime as pickup_datetime, tpep_dropoff_datetime as dropoff_datetime, PULocationID as LocationID, total_amount, pickup_zone FROM taxi_data_clean

In [0]:
create or replace table taxi_labs.new_data.view_gold as select pickup_zone, round(avg(total_amount), 2) as average_cost, count(locationID) as total_trips
         from silver_view  
          group by locationID, pickup_zone
          having total_trips > 100
          order by average_cost desc
          limit 10

